In [1]:
print("all ok")

all ok


In [7]:
import os
from dotenv import load_dotenv
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.memory import MemoryContent,Memory,MemoryMimeType
from autogen_agentchat.ui import Console
from autogen_ext.memory.chromadb import ChromaDBVectorMemory,PersistentChromaDBVectorMemoryConfig
from autogen_ext.memory.chromadb import SentenceTransformerEmbeddingFunctionConfig
from pathlib import Path
import re
from typing import List
import aiofiles
import aiohttp

load_dotenv()

True

In [3]:
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [4]:
class SimpleDocumentIndexer:
    """Basic document indexer for Autogen memory"""
    def __init__(self,memory:Memory,chunk_size:int=1000) -> None:
        self.memory=memory
        self.chunk_size=chunk_size
    async def _fetch_content(self,source:str) -> str:
        """Fetch content from URL or files"""
        if source.startswith(('http://,https://')):
            async with aiohttp.ClientSession() as session:
                async with session.get(source) as response:
                    return await response.text()
        else:
            async with aiofiles.open(source,'r',encoding='utf-8') as f:
                return await f.read()
            
    def _strip_html(self,text:str)->str:
        """Remove HTML tags and normalize whitespace."""
        text=re.sub(r"<[^>]*>"," ",text)
        text=re.sub(r"\s+"," ",text)
        return text.strip()
    
    def _split_text(self,text:str) -> List[str]:
        """Split text into fixed-size chunks."""
        chunks: list[str] = []
        # Just split text into fixed-size chunks
        for i in range(0, len(text), self.chunk_size):
            chunk = text[i : i + self.chunk_size]
            chunks.append(chunk.strip())
        return chunks
    async def index_documents(self, sources: List[str]) -> int:
        """Index documents into memory."""
        total_chunks = 0

        for source in sources:
            try:
                content = await self._fetch_content(source)

                # Strip HTML if content appears to be HTML
                if "<" in content and ">" in content:
                    content = self._strip_html(content)

                chunks = self._split_text(content)

                for i, chunk in enumerate(chunks):
                    await self.memory.add(
                        MemoryContent(
                            content=chunk, mime_type=MemoryMimeType.TEXT, metadata={"source": source, "chunk_index": i}
                        )
                    )

                total_chunks += len(chunks)

            except Exception as e:
                print(f"Error indexing {source}: {str(e)}")
        return total_chunks

In [9]:
rag_memory=ChromaDBVectorMemory(
    config=PersistentChromaDBVectorMemoryConfig(
        collection_name="autogen_docs",
        persistence_path=os.path.join(str(Path.home()), ".chromadb_autogen"),
        k=5,  # Return top 3 results
        score_threshold=0.4,  # Minimum similarity score
        
    )
    )
await rag_memory.clear()

In [10]:
async def index_autogen_docs() -> None:
    indexer = SimpleDocumentIndexer(memory=rag_memory)
    sources = [
        "https://microsoft.github.io/autogen/dev/user-guide/agentchat-user-guide/tutorial/agents.html"
        
    ]
    chunks: int = await indexer.index_documents(sources)
    print(f"Indexed {chunks} chunks from {len(sources)} AutoGen documents")


await index_autogen_docs()

Error indexing https://microsoft.github.io/autogen/dev/user-guide/agentchat-user-guide/tutorial/agents.html: [Errno 22] Invalid argument: 'https://microsoft.github.io/autogen/dev/user-guide/agentchat-user-guide/tutorial/agents.html'
Indexed 0 chunks from 1 AutoGen documents


In [11]:
# Create our RAG assistant agent
rag_assistant = AssistantAgent(
    name="rag_assistant", model_client=OpenAIChatCompletionClient(model="gpt-4o"), memory=[rag_memory]
)

In [12]:
# Ask questions about AutoGen
stream = rag_assistant.run_stream(task="What is AgentChat?")
await Console(stream)

# Remember to close the memory when done
await rag_memory.close()

---------- TextMessage (user) ----------
What is AgentChat?


C:\Users\Hp\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:29<00:00, 2.77MiB/s]


---------- TextMessage (rag_assistant) ----------
AgentChat is a term that can refer to different systems or solutions designed for enabling communication between users and AI agents such as virtual assistants, chatbots, or customer service bots. These systems are typically aimed at improving customer engagement, automating support, or facilitating human-computer interaction.

1. **AI-powered Chatbots:** AgentChat might involve AI-driven chatbots that interact with users on websites or apps to answer questions, provide assistance, and guide users through processes. These chatbots aim to enhance user experience by providing quick responses and reducing the need for human intervention.

2. **Customer Support Platforms:** In customer service, AgentChat platforms can connect customers with live agents or virtual assistants for real-time support. They often include features like chatbot handover to human agents, conversation analytics, and support for multiple communication channels.

3. **

In [13]:
# Ask questions about AutoGen
stream = rag_assistant.run_stream(task="What is AssistantAgent?")
await Console(stream)

# Remember to close the memory when done
await rag_memory.close()

---------- TextMessage (user) ----------
What is AssistantAgent?
---------- TextMessage (rag_assistant) ----------
"AssistantAgent" is not a universally recognized term or specific product, so it may refer to various concepts depending on the context in which it is used. Here are a few possibilities:

1. **Virtual Assistant Software:** AssistantAgent could refer to software or applications designed to act as virtual assistants. These agents perform tasks, answer questions, and assist users through natural language processing and other AI technologies. Examples include Siri, Google Assistant, and Amazon Alexa.

2. **Customer Service Agents:** In a customer service context, AssistantAgent might denote systems or software that help human agents perform their duties more efficiently. This could involve AI tools that suggest responses to customer queries or manage routine tasks.

3. **AI-Powered Tools:** The term could also describe AI-powered tools across various industries designed to ass